In [34]:
from typing import Tuple

from numpy import arange, NaN
from tensorflow import Tensor
from tensorflow.math import is_nan
from tensorflow.python.data import Dataset

from tfdatacompose import Map, Filter, Pipeline, Skip, Take, Print

In [35]:
class RemoveNone(Filter):
    def filter(self, number: int) -> bool:
        return not is_nan(number)

class RemoveOutOfRange(Filter):
    def __init__(self, min, max):
        self.min = min
        self.max = max
        
    def filter(self, number: int) -> bool:
        return self.min < number < self.max

class FilterSign(Filter):
    def __init__(self, positive):
        self.positive = positive
    
    def filter(self, number: int) -> bool:
        if self.positive:
            return 0 <= number
        else:
            return number <= 0

class FilterOddness(Filter):
    def __init__(self, even):
        self.even = even
    
    def filter(self, number: int) -> bool:
        if self.even:
            return number % 2 == 0 
        else:
            return number % 2 == 1


In [37]:
dataset = Dataset.from_tensor_slices([-10,-9,NaN,-8,-7,-6,-5,NaN,-4,-3,-2,-1,0,1,NaN,2,3,4,5,6,7,8,9,10])
print(list(dataset.as_numpy_iterator())) # [-10.0, -9.0, nan, -8.0, -7.0, -6.0, -5.0, nan, -4.0, -3.0, -2.0, -1.0, 0.0, 1.0, nan, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]

class CleanData(Pipeline):
    def __init__(self):
        super().__init__([
            RemoveNone(),
            RemoveOutOfRange(-5, 5),
        ])

positiveOdd = Pipeline([
    CleanData(),
    FilterSign(positive=True),
    FilterOddness(even=False)
])(dataset)
print(list(positiveOdd.as_numpy_iterator())) # [1.0, 3.0]

negativeEven = Pipeline([
    CleanData(),
    FilterSign(positive=False),
    FilterOddness(even=True)
])(dataset)
print(list(negativeEven.as_numpy_iterator())) # [-4.0, -2.0, 0.0]

[-10.0, -9.0, nan, -8.0, -7.0, -6.0, -5.0, nan, -4.0, -3.0, -2.0, -1.0, 0.0, 1.0, nan, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]
[1.0, 3.0]
[-4.0, -2.0, 0.0]
